# Text-to-SQL RL with Verifiable Rewards

This notebook is a guided version of the RL half of `texttosql_sft_grpo.py`.

The recipe optimizes a simple behavior: given a schema and a question, generate SQL that returns the right rows when executed. The client script owns the task-specific parts: it builds prompts, runs SQLite, turns execution results into rewards, and assembles the training batch. Open-RL owns the model side: sampling from the current adapter, running `forward_backward`, and stepping the optimizer.

We keep the run small so the recipe is easy to inspect. The full script is still the path for longer runs and final curves.


## Recipe Flow

Each RL step in this notebook mirrors the script:

1. choose a few Text-to-SQL prompts
2. save the current adapter weights for the sampler
3. sample multiple SQL candidates per prompt
4. execute each candidate in SQLite and score it
5. build completion-weighted PPO datums from rewards and sample logprobs
6. send the batch to Open-RL for the update

The useful thing to watch is not the algorithm name. It is the data moving across the boundary: SQL strings and rewards stay in the client; tokens, logprobs, gradients, and optimizer state live in Open-RL.


## Prerequisites

Start Open-RL before running the server-backed cells. For Gemma 4 E2B, use the setup in this directory's `README.md`: vLLM sampler on one GPU, gateway/trainer on another GPU.

The notebook is safe to open: data loading is on by default, but server calls and training are behind flags in the config cell.


In [ ]:
from __future__ import annotations

import asyncio
import os
import random
import re
import sqlite3
import statistics
import time
from collections.abc import Sequence
from typing import Any, cast

import matplotlib.pyplot as plt
import pandas as pd
import tinker
from datasets import load_dataset
from tinker import types
from transformers import AutoTokenizer


In [ ]:
BASE_URL = os.getenv("TINKER_BASE_URL", "http://127.0.0.1:9003")
BASE_MODEL = "google/gemma-4-e2b"
DATASET_NAME = "philschmid/gretel-synthetic-text-to-sql"

RUN_LOAD_DATA = True
RUN_BASELINE = False
RUN_RL = False

config = {
    "base_url": BASE_URL,
    "model": BASE_MODEL,
    "rank": 32,
    "seed": 42,
    "dataset_limit": 2_000,
    "train_limit": 96,
    "eval_limit": 24,
    "steps": 8,
    "prompts_per_step": 4,
    "samples_per_prompt": 4,
    "max_tokens": 64,
    "temperature": 0.8,
    "learning_rate": 5e-6,
    "clip_range": 0.2,
    "kl_coeff": 0.1,
}

pd.DataFrame.from_dict(config, orient="index", columns=["value"])


## Step 1: Make a Prompt

The model should output only a SQLite query. The schema is part of the prompt because every example uses a different small database.


In [ ]:
PROMPT_TEMPLATE = """Return only one SQLite query.

Schema:
{schema}

Question:
{question}

SQL:
"""


def make_prompt(schema: str, question: str) -> str:
    return PROMPT_TEMPLATE.format(schema=schema, question=question)


## Step 2: Execute SQL for Reward

Execution is the verifiable part. We create an in-memory SQLite database from the schema and inserts, run the generated query, and compare rows to the target query.


In [ ]:
def clean_sql(text: str) -> str:
    text = re.sub(r"<think>.*?</think>", " ", text, flags=re.DOTALL)
    text = re.sub(r"^```(?:sql)?|```$", "", text.strip(), flags=re.IGNORECASE)
    match = re.search(r"\b(with|select|insert|update|delete)\b", text, flags=re.IGNORECASE)
    if match:
        text = text[match.start() :]
    return text.strip().rstrip(";")


def normalize_sql(text: str) -> str:
    return " ".join(clean_sql(text).lower().split())


def run_sql(schema_and_data: str, query: str) -> tuple[list[tuple[Any, ...]] | None, str | None]:
    connection = sqlite3.connect(":memory:")
    try:
        deadline = time.monotonic() + 0.25
        connection.set_progress_handler(lambda: 1 if time.monotonic() > deadline else 0, 10_000)
        connection.executescript(schema_and_data)
        rows = connection.execute(clean_sql(query)).fetchall()
        return sorted(rows, key=repr), None
    except sqlite3.Error as error:
        return None, str(error)
    finally:
        connection.close()


In [ ]:
def score_sql(predicted_sql: str, example: dict[str, Any]) -> dict[str, Any]:
    predicted_rows, error = run_sql(example["schema"], predicted_sql)
    target_rows = example["target_rows"]
    compiles = error is None
    execution_match = compiles and predicted_rows == target_rows
    exact_match = normalize_sql(predicted_sql) == normalize_sql(example["target_sql"])

    reward = 0.0
    reward += 0.25 if compiles else -0.25
    reward += 2.0 if execution_match else 0.0
    reward += 0.25 if exact_match else 0.0

    return {
        "predicted_sql": clean_sql(predicted_sql),
        "reward": reward,
        "compile": float(compiles),
        "execution_match": float(execution_match),
        "exact_match": float(exact_match),
        "sqlite_error": error or "",
    }


### Reward Sanity Check

Before using a model, score two hand-written queries. The correct query receives the execution reward; the wrong query can still compile, but it does not return the target rows.


In [ ]:
toy_example = {
    "schema": """
CREATE TABLE employees (name TEXT, department TEXT, salary INTEGER);
INSERT INTO employees VALUES ('Ada', 'Research', 300), ('Grace', 'Platform', 250);
""",
    "question": "Which employee has the highest salary?",
    "target_sql": "SELECT name FROM employees ORDER BY salary DESC LIMIT 1",
}
toy_example["target_rows"], _ = run_sql(toy_example["schema"], toy_example["target_sql"])

pd.DataFrame(
    [
        {"candidate": "correct", **score_sql("SELECT name FROM employees ORDER BY salary DESC LIMIT 1", toy_example)},
        {"candidate": "wrong", **score_sql("SELECT department FROM employees ORDER BY salary ASC LIMIT 1", toy_example)},
    ]
)


## Step 3: Load Executable Text-to-SQL Rows

The dataset has many examples, but RL reward only works cleanly when the target query executes and returns rows. We filter to those examples and then tokenize the prompt.


In [ ]:
def load_examples(config: dict[str, Any], tokenizer: Any):
    dataset = load_dataset(DATASET_NAME, split="train").shuffle(seed=config["seed"])
    dataset = dataset.select(range(min(config["dataset_limit"], len(dataset))))
    split = dataset.train_test_split(test_size=min(500, len(dataset) // 5), shuffle=False)

    def collect(rows, limit: int) -> list[dict[str, Any]]:
        examples = []
        for row in rows:
            schema = row["sql_context"]
            target_sql = clean_sql(row["sql"])
            target_rows, error = run_sql(schema, target_sql)
            if error or not target_rows:
                continue

            prompt_text = make_prompt(schema, row["sql_prompt"])
            prompt_tokens = tokenizer.encode(prompt_text, add_special_tokens=False)
            if len(prompt_tokens) >= 512:
                continue

            examples.append(
                {
                    "schema": schema,
                    "question": row["sql_prompt"],
                    "target_sql": target_sql,
                    "target_rows": target_rows,
                    "prompt_text": prompt_text,
                    "prompt_tokens": prompt_tokens,
                }
            )
            if len(examples) >= limit:
                break
        return examples

    return collect(split["train"], config["train_limit"]), collect(split["test"], config["eval_limit"])


In [ ]:
if RUN_LOAD_DATA:
    tokenizer = AutoTokenizer.from_pretrained(config["model"])
    train_examples, eval_examples = load_examples(config, tokenizer)
else:
    tokenizer = None
    train_examples = []
    eval_examples = []

len(train_examples), len(eval_examples)


In [ ]:
if train_examples:
    example = train_examples[0]
    print(example["prompt_text"][:800])
    print("Target SQL:", normalize_sql(example["target_sql"]))
    print("Target rows:", example["target_rows"][:3])


## Step 4: Convert Rewards into Advantages

For each prompt, we sample a group of completions. Rewards become advantages by subtracting the group mean and dividing by the group standard deviation. Constant-reward groups are skipped because they provide no preference signal.


In [ ]:
def group_advantages(rewards: list[float]) -> list[float]:
    if len(rewards) < 2:
        return [0.0] * len(rewards)
    std = statistics.pstdev(rewards)
    if std < 1e-8:
        return [0.0] * len(rewards)
    mean = statistics.fmean(rewards)
    return [(reward - mean) / std for reward in rewards]


toy_rewards = [0.25, 2.5, 0.25, 1.0]
pd.DataFrame({"reward": toy_rewards, "advantage": group_advantages(toy_rewards)})


## Step 5: Build the PPO Datum

The PPO loss should only apply to completion tokens. Prompt tokens are context, so their weights are zero. Completion tokens get the group-relative advantage and the old logprobs from sampling.


In [ ]:
def make_rl_datum(
    prompt_tokens: list[int],
    completion_tokens: list[int],
    completion_logprobs: list[float],
    advantage: float,
) -> types.Datum:
    tokens = prompt_tokens + completion_tokens
    prompt_padding = [0.0] * (len(prompt_tokens) - 1)
    completion_weights = [1.0] * len(completion_tokens)

    return types.Datum(
        model_input=types.ModelInput.from_ints(tokens=tokens[:-1]),
        loss_fn_inputs=cast(
            Any,
            {
                "target_tokens": tokens[1:],
                "weights": prompt_padding + completion_weights,
                "logprobs": prompt_padding + completion_logprobs,
                "advantages": prompt_padding + [advantage] * len(completion_tokens),
            },
        ),
    )


## Step 6: Evaluate the Current Policy

Evaluation is just sampling plus reward aggregation. We use temperature zero to make the metric less noisy.


In [ ]:
async def evaluate_sampler(sampler: Any, examples: list[dict[str, Any]]) -> dict[str, float]:
    scores = []
    futures = [
        sampler.sample_async(
            prompt=types.ModelInput.from_ints(tokens=example["prompt_tokens"]),
            num_samples=1,
            sampling_params=types.SamplingParams(max_tokens=config["max_tokens"], temperature=0.0, seed=config["seed"] + index),
        )
        for index, example in enumerate(examples)
    ]
    responses = await asyncio.gather(*futures)

    for example, response in zip(examples, responses):
        tokens = response.sequences[0].tokens if response.sequences else []
        predicted_sql = tokenizer.decode(tokens, skip_special_tokens=True)
        scores.append(score_sql(predicted_sql, example))

    if not scores:
        return {"compile": 0.0, "execution_match": 0.0, "exact_match": 0.0, "reward": 0.0}
    return {
        key: statistics.fmean(float(score[key]) for score in scores)
        for key in ["compile", "execution_match", "exact_match", "reward"]
    }


In [ ]:
baseline_metrics = {}

if RUN_BASELINE:
    service_client = tinker.ServiceClient(api_key="tml-dummy-key", base_url=config["base_url"])
    await service_client.get_server_capabilities_async()
    trainer = await service_client.create_lora_training_client_async(
        base_model=config["model"],
        rank=config["rank"],
        seed=config["seed"],
        train_mlp=True,
        train_attn=True,
        train_unembed=False,
    )
    sampler = await trainer.save_weights_and_get_sampling_client_async(name="texttosql_notebook_baseline")
    baseline_metrics = await evaluate_sampler(sampler, eval_examples)

baseline_metrics


## Step 7: The RL Update Step

This cell is the heart of the recipe. It snapshots weights, samples SQL, scores SQL locally, builds PPO datums, and asks Open-RL to run the GPU update.


In [ ]:
async def grpo_step(trainer: Any, examples: list[dict[str, Any]], step: int) -> tuple[dict[str, float], str]:
    sampler = await trainer.save_weights_and_get_sampling_client_async(name=f"texttosql_notebook_rollout_s{step}")
    sampling_params = types.SamplingParams(max_tokens=config["max_tokens"], temperature=config["temperature"])

    futures = [
        sampler.sample_async(
            prompt=types.ModelInput.from_ints(tokens=example["prompt_tokens"]),
            num_samples=config["samples_per_prompt"],
            sampling_params=sampling_params,
        )
        for example in examples
    ]
    responses = await asyncio.gather(*futures)

    datums = []
    rollouts = []
    for example, response in zip(examples, responses):
        group = []
        for sequence in response.sequences:
            predicted_sql = tokenizer.decode(sequence.tokens, skip_special_tokens=True)
            score = score_sql(predicted_sql, example)
            group.append(
                {
                    **score,
                    "question": example["question"],
                    "target_sql": example["target_sql"],
                    "prompt_tokens": example["prompt_tokens"],
                    "completion_tokens": list(sequence.tokens),
                    "completion_logprobs": [float(value) for value in (sequence.logprobs or [])],
                }
            )

        advantages = group_advantages([float(item["reward"]) for item in group])
        for rollout, advantage in zip(group, advantages):
            if abs(advantage) < 1e-8:
                continue
            if len(rollout["completion_tokens"]) != len(rollout["completion_logprobs"]):
                continue

            rollouts.append({**rollout, "advantage": advantage})
            datums.append(
                make_rl_datum(
                    rollout["prompt_tokens"],
                    rollout["completion_tokens"],
                    rollout["completion_logprobs"],
                    advantage,
                )
            )

    if datums:
        fwdbwd_future = await trainer.forward_backward_async(
            datums,
            "ppo",
            loss_fn_config={"clip_range": config["clip_range"], "kl_coeff": config["kl_coeff"]},
        )
        optim_future = await trainer.optim_step_async(types.AdamParams(learning_rate=config["learning_rate"]))
        fwdbwd = await fwdbwd_future
        await optim_future
        loss = float(fwdbwd.metrics.get("loss:mean", 0.0))
    else:
        loss = 0.0

    if rollouts:
        metrics = {
            "loss": loss,
            "reward": statistics.fmean(float(item["reward"]) for item in rollouts),
            "compile": statistics.fmean(float(item["compile"]) for item in rollouts),
            "execution_match": statistics.fmean(float(item["execution_match"]) for item in rollouts),
            "num_rollouts": float(len(rollouts)),
        }
        best = max(rollouts, key=lambda item: (item["reward"], item["execution_match"]))
        note = (
            f"question: {best['question']}\n"
            f"predicted: {normalize_sql(best['predicted_sql'])}\n"
            f"target: {normalize_sql(best['target_sql'])}\n"
            f"reward: {best['reward']:.2f}, advantage: {best['advantage']:.2f}"
        )
    else:
        metrics = {"loss": loss, "reward": 0.0, "compile": 0.0, "execution_match": 0.0, "num_rollouts": 0.0}
        note = "No non-zero-advantage rollouts this step."

    return metrics, note


## Step 8: Run a Short Loop

Turn `RUN_RL` on in the config cell when the server is ready. The loop below is intentionally short; it is for learning and debugging, not for producing the final curve.


In [ ]:
training_records = []
rollout_notes = []

if RUN_RL:
    rng = random.Random(config["seed"])
    service_client = tinker.ServiceClient(api_key="tml-dummy-key", base_url=config["base_url"])
    await service_client.get_server_capabilities_async()
    trainer = await service_client.create_lora_training_client_async(
        base_model=config["model"],
        rank=config["rank"],
        seed=config["seed"],
        train_mlp=True,
        train_attn=True,
        train_unembed=False,
    )

    for step in range(1, config["steps"] + 1):
        batch = rng.sample(train_examples, k=min(config["prompts_per_step"], len(train_examples)))
        metrics, note = await grpo_step(trainer, batch, step)
        training_records.append({"phase": "rl_train", "step": step, **metrics})
        rollout_notes.append(f"step {step}\n{note}")
        print(
            f"step {step:03d}: reward={metrics['reward']:.3f} "
            f"exec={metrics['execution_match'] * 100:.1f}% "
            f"compile={metrics['compile'] * 100:.1f}% "
            f"rollouts={metrics['num_rollouts']:.0f}"
        )

metrics_df = pd.DataFrame(training_records)
metrics_df


## Read the Curve

On tiny runs, the rollout metrics matter more than final eval. Look for valid SQL first (`compile`), then execution match. A noisy line is expected because every step samples fresh completions.


In [ ]:
if not metrics_df.empty:
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
    axes[0].plot(metrics_df["step"], metrics_df["reward"], marker="o")
    axes[0].set_title("reward")
    axes[1].plot(metrics_df["step"], metrics_df["execution_match"], marker="o")
    axes[1].set_title("execution match")
    axes[2].plot(metrics_df["step"], metrics_df["compile"], marker="o")
    axes[2].set_title("compile rate")
    for axis in axes:
        axis.set_xlabel("step")
        axis.grid(alpha=0.25)
    plt.show()


In [ ]:
if rollout_notes:
    print("\n---\n".join(rollout_notes[-3:]))


## Connect Back to the Full Recipe

This notebook keeps the loop small. The script version adds the production pieces around the same core idea:

- optional SFT warm start
- larger train/eval splits
- shaped reward terms beyond the simple execution reward shown here
- checkpointing of final states
- metrics logging and plotting utilities

Once this notebook makes sense, run `texttosql_sft_grpo.py` from the README for the full recipe.
